### Assignment-03

- Using the provided ```U-Net``` training notebook as a reference, develop and train a ```DeepLabv3+``` model on the sample dataset available in the ```data``` folder (accessible through the provided Google Drive shortcut).

- For model evaluation, use the same test image that was previously used to evaluate the U-Net model. Save and upload the output image generated by your trained DeepLabv3+ model to the ```data/output``` folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import torchvision
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import os

# Config
IMG_SIZE = 128
BATCH_SIZE = 8  # DeepLabv3+ is heavier
EPOCHS = 5
LR = 0.0001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = '/content/data/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Dataset
class SimpleDataset(Dataset):
    def __init__(self, images_dir, masks_dir):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor()
        ])

        imgs = list(self.images_dir.glob('*.jpg'))
        self.pairs = [img for img in imgs
                     if (self.masks_dir / f"{img.stem}_mask.png").exists()]
        print(f" Dataset ready: {len(self.pairs)} pairs")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path = self.pairs[idx]
        mask_path = self.masks_dir / f"{img_path.stem}_mask.png"

        img = cv2.imread(str(img_path))[..., ::-1]  # BGR→RGB
        mask = cv2.imread(str(mask_path), 0)
        mask = (mask > 127).astype(np.float32)

        img = self.transform(img)
        mask = torch.tensor(cv2.resize(mask, (IMG_SIZE, IMG_SIZE)), dtype=torch.float32).unsqueeze(0)

        return img, mask

# Load dataset
dataset = SimpleDataset(
    '/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017_SAMPLE/train2017',
    '/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017_SAMPLE/mask_train2017'
)
train_loader = DataLoader(dataset, BATCH_SIZE, shuffle=True, num_workers=2)

# Model
model = torchvision.models.segmentation.deeplabv3_resnet50(pretrained=False, num_classes=1)
model = model.to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss()  # DeepLabv3+ outputs logits

print(f"🚀 Training {len(dataset)} images")

# Training loop
for epoch in range(EPOCHS):
    model.train()
    loss_total = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for imgs, masks in pbar:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs)['out']
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        loss_total += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    print(f"Epoch {epoch+1} Avg Loss: {loss_total/len(train_loader):.4f}")

# Save model
torch.save(model.state_dict(), 'deeplabv3_seg.pth')
print("✅ Saved DeepLabv3+ model!")

# Inference function
def remove_bg_deeplab(img_path, model_path='deeplabv3_seg.pth'):
    model = torchvision.models.segmentation.deeplabv3_resnet50(pretrained=False, num_classes=1)
    model.load_state_dict(torch.load(model_path))
    model.to(DEVICE).eval()

    img = cv2.imread(img_path)
    orig = img.copy()
    h, w = img.shape[:2]

    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor()
    ])
    img_tensor = transform(img[..., ::-1]).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        mask = torch.sigmoid(model(img_tensor)['out'])[0,0].cpu().numpy()
        mask = cv2.resize(mask, (w, h)) > 0.5

    result = orig * mask[:,:,None]
    save_path = os.path.join(OUTPUT_DIR, 'result_no_bg_deeplab.jpg')
    cv2.imwrite(save_path, result)
    print(f"🎯 Saved: {save_path}")
    return save_path

# Test
test_img = "/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017_SAMPLE/train2017/000000532933.jpg"
output_path = remove_bg_deeplab(test_img)

# Download
from google.colab import files
files.download(output_path)


 Dataset ready: 3768 pairs


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 153MB/s]


🚀 Training 3768 images


...

Epoch 4 Avg Loss: 0.2527


Epoch 5: 100%|██████████| 471/471 [01:42<00:00,  4.59it/s, loss=0.2209]


Epoch 5 Avg Loss: 0.2181
✅ Saved DeepLabv3+ model!
🎯 Saved: /content/data/output/result_no_bg_deeplab.jpg


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>